# Meeting Intelligence & Decision Extractor (free, Hugging Face only)

Turn a messy meeting transcript into **structured JSON**: summary, decisions, action items (owner, task, due date) and open questions.

**What you will learn along the way**
1. Attention intuition, tokens and context windows
2. Temperature and top-p
3. Latency and cost (local vs hosted)
4. System vs user prompts, few-shot prompting
5. Structured JSON output and validation
6. Prompt versioning and regression tests
7. Experiments: context length, extraction accuracy, JSON validity, hosted vs local

**Dependencies:** `transformers`, `torch`, `huggingface_hub`, `numpy`. No paid APIs.
The hosted part uses Hugging Face's free Inference tier with a free token (optional, the notebook skips it if absent).

## 0. Setup

In [1]:
import importlib
for name in ["torch", "transformers", "huggingface_hub", "numpy", "ipykernel"]:
    try:
        m = importlib.import_module(name)
        print(f"OK       {name} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"MISSING  {name}")

OK       torch 2.9.0+cpu
OK       transformers 5.20.1
OK       huggingface_hub 2.2.0
OK       numpy 1.26.4
OK       ipykernel 6.29.5


In [2]:
import os, re, json, time, random
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---- Settings you can change ----
LOCAL_MODEL  = "Qwen/Qwen2.5-0.5B-Instruct"   # small, runs on a laptop CPU (~1 GB download)
HOSTED_MODEL = "Qwen/Qwen2.5-7B-Instruct"     # bigger model, run on HF's servers
HF_TOKEN     = os.environ.get("HF_TOKEN")     # free token: https://huggingface.co/settings/tokens
QUICK        = True                           # True = fewer runs so the notebook finishes fast

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| Hosted available:", bool(HF_TOKEN))

Device: cpu | Hosted available: False


## 1. Sample meetings with "gold" answers

Each meeting has hand-written gold items. A gold item is a list of keywords that must all appear in an extracted item.
This keeps scoring simple and transparent, with no extra libraries.

In [3]:
MEETINGS = {
"product_sync": {
"transcript": """Alice (PM): Let's start with the launch. QA found two blockers, so we can't ship on November 1.
Bob (Eng): Agreed. I'd propose November 15 instead.
Alice (PM): Okay, decision made: launch moves to November 15.
Bob (Eng): On the database, Mongo is giving us trouble with reporting queries. I want to switch to Postgres.
Alice (PM): Fine, we'll use Postgres. Bob, can you write the migration plan by next Wednesday?
Priya (Design): I'll finalize the pricing page mockups by Friday.
Alice (PM): Great. One open question: do we need legal review for the new terms? Nobody knows yet.
Alice (PM): I'll email the legal team today to ask.""",
"gold_decisions": [["november 15"], ["postgres"]],
"gold_actions":   [["bob", "migration"], ["priya", "pricing"], ["alice", "legal"]],
},
"budget_review": {
"transcript": """Dana (Manager): Budget check. Marketing wants $20k for the conference booth.
Sam (Marketing): We think the booth will bring in leads.
Dana (Manager): That's too much this quarter. We approve $8k, a smaller table only.
Lee (Ops): Related, the office move is postponed until January because the lease isn't signed.
Dana (Manager): Right, that's decided. Sam, send me the revised booth plan by Tuesday.
Lee (Ops): I'll call the landlord tomorrow and get the lease timeline.
Dana (Manager): Open item: who covers travel costs? We'll revisit next week.""",
"gold_decisions": [["booth", "8"], ["move", "january"]],
"gold_actions":   [["sam", "booth"], ["lee", "landlord"]],
},
}

# A tiny meeting used ONLY as a few-shot example (never scored)
EXAMPLE_TRANSCRIPT = """Maya (Lead): We'll switch the weekly standup to Mondays. That's final.
Tom (Dev): I'll update the calendar invite by tomorrow.
Maya (Lead): Should we also change the retro day? Let's discuss next time."""
EXAMPLE_JSON = {
    "summary": "The team moved the weekly standup to Mondays.",
    "decisions": [{"decision": "Weekly standup moves to Mondays", "owner": "Maya"}],
    "action_items": [{"owner": "Tom", "task": "Update the calendar invite", "due": "tomorrow"}],
    "open_questions": ["Should the retro day change?"],
}

# Off-topic chatter we use to make the context longer (needle-in-a-haystack test)
FILLER = [
 "Sam (Marketing): Did anyone watch the game last night? The second half was wild.",
 "Lee (Ops): The coffee machine on the third floor is broken again.",
 "Priya (Design): I found a great font pack, I'll share the link later, no rush.",
 "Bob (Eng): My train was late this morning, so sorry if I seem sleepy.",
 "Dana (Manager): Reminder that the fire drill is sometime next month.",
 "Alice (PM): The new snack selection is pretty good, honestly.",
]

def build_context(transcript, level):
    """level: 'short' (as is), 'medium' (~+500 tokens chatter), 'long' (~+1500 tokens)."""
    reps = {"short": 0, "medium": 6, "long": 20}[level]
    chatter = [FILLER[i % len(FILLER)] for i in range(reps * len(FILLER) // 2)]
    half = len(chatter) // 2
    return "\n".join(chatter[:half] + [transcript] + chatter[half:])

print({k: len(v["transcript"].split()) for k, v in MEETINGS.items()}, "words")

{'product_sync': 109, 'budget_review': 88} words


## 2. Tokens, context windows and attention intuition

**Tokens**: models read sub-word pieces, not words. Cost, speed and the context limit are all measured in tokens.

**Context window**: the max tokens (prompt + answer) the model can see at once. Long meetings must fit.

**Attention**: for each token, the model computes how much to "look at" every other token, using
`softmax(Q·Kᵀ / √d) · V`. Below is a toy version.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
text = MEETINGS["product_sync"]["transcript"]

n_words  = len(text.split())
n_tokens = len(tokenizer(text)["input_ids"])
print(f"Words: {n_words} | Tokens: {n_tokens} | tokens/word: {n_tokens / n_words:.2f}")
print("Example split:", tokenizer.tokenize("Postgres migration by Wednesday")[:10])

ctx = tokenizer.model_max_length
print(f"Declared context window: {ctx:,} tokens")
for level in ["short", "medium", "long"]:
    n = len(tokenizer(build_context(text, level))["input_ids"])
    print(f"  {level:6s}: {n:5d} tokens  ({n / ctx:6.2%} of window)")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Words: 109 | Tokens: 162 | tokens/word: 1.49
Example split: ['Post', 'gres', 'Ġmigration', 'Ġby', 'ĠWednesday']
Declared context window: 131,072 tokens
  short :   162 tokens  ( 0.12% of window)
  medium:   468 tokens  ( 0.36% of window)
  long  :  1182 tokens  ( 0.90% of window)


In [5]:
# Toy self-attention: 5 tokens, hand-made 3-d vectors, so you can SEE who attends to whom.
words = ["the", "report", "was", "late", "it"]
# Dimensions loosely mean: [is-a-noun-thing, is-a-state/verb, is-a-pronoun]
X = np.array([[0.0, 0.0, 0.0],   # the
              [1.0, 0.0, 0.0],   # report  (a thing)
              [0.0, 1.0, 0.0],   # was
              [0.0, 1.0, 0.2],   # late
              [0.5, 0.0, 1.0]])  # it      (pronoun looking for a thing)

Wq = np.array([[0,0,0],[0,0,0],[2.5,0,0]], dtype=float)   # pronoun asks: "which token is a thing?"
Wk = np.array([[2.0,0,0],[0,0.3,0],[0,0,0.3]])            # nouns advertise "I am a thing"
Q, K = X @ Wq, X @ Wk
scores  = Q @ K.T / np.sqrt(K.shape[1])
weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)   # softmax per row

print("Attention weights (row = token looking, column = token looked at)\n")
print(" " * 8 + "".join(f"{w:>8s}" for w in words))
for w, row in zip(words, weights):
    print(f"{w:>8s}" + "".join(f"{v:8.2f}" for v in row))
print("\nRead the last row: 'it' puts its largest weight on 'report'.")

print("\nWhy long contexts get slow: attention compares every token with every token (n^2).")
for n in [500, 2000, 8000, 32000]:
    print(f"  {n:6d} tokens -> {n*n:>14,} pairs")

Attention weights (row = token looking, column = token looked at)

             the  report     was    late      it
     the    0.20    0.20    0.20    0.20    0.20
  report    0.20    0.20    0.20    0.20    0.20
     was    0.20    0.20    0.20    0.20    0.20
    late    0.16    0.29    0.16    0.16    0.22
      it    0.04    0.71    0.04    0.04    0.17

Read the last row: 'it' puts its largest weight on 'report'.

Why long contexts get slow: attention compares every token with every token (n^2).
     500 tokens ->        250,000 pairs
    2000 tokens ->      4,000,000 pairs
    8000 tokens ->     64,000,000 pairs
   32000 tokens ->  1,024,000,000 pairs


## 3. Model backends: local and hosted

Both backends return the same dict (`text`, `latency_s`, `prompt_tokens`, `completion_tokens`, `error`) so we can compare them fairly.

**Sampling controls**
- **temperature**: 0 means pick the most likely token (repeatable). Higher means more random.
- **top_p**: only sample from the smallest set of tokens whose probabilities add up to p.

For extraction we usually want low temperature.

In [6]:
_local = {}

def load_local():
    if "model" not in _local:
        t0 = time.time()
        _local["model"] = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL, torch_dtype="auto").to(DEVICE)
        _local["model"].eval()
        print(f"Loaded {LOCAL_MODEL} in {time.time() - t0:.1f}s")
    return _local["model"]

def call_local(messages, temperature=0.0, top_p=1.0, max_new_tokens=400, seed=0):
    model = load_local()
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    torch.manual_seed(seed)
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        kwargs.update(do_sample=False)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    latency = time.time() - t0
    new_ids = out[0][inputs["input_ids"].shape[1]:]
    return {"text": tokenizer.decode(new_ids, skip_special_tokens=True), "latency_s": latency,
            "prompt_tokens": inputs["input_ids"].shape[1], "completion_tokens": len(new_ids), "error": None}

def call_hosted(messages, temperature=0.0, top_p=1.0, max_new_tokens=400, seed=0):
    if not HF_TOKEN:
        return {"text": "", "latency_s": 0, "prompt_tokens": 0, "completion_tokens": 0, "error": "no HF_TOKEN"}
    from huggingface_hub import InferenceClient
    client = InferenceClient(model=HOSTED_MODEL, token=HF_TOKEN)
    t0 = time.time()
    try:
        r = client.chat_completion(messages=messages, max_tokens=max_new_tokens,
                                   temperature=max(temperature, 0.01), top_p=top_p, seed=seed)
        u = r.usage
        return {"text": r.choices[0].message.content, "latency_s": time.time() - t0,
                "prompt_tokens": getattr(u, "prompt_tokens", 0), "completion_tokens": getattr(u, "completion_tokens", 0),
                "error": None}
    except Exception as e:
        return {"text": "", "latency_s": time.time() - t0, "prompt_tokens": 0, "completion_tokens": 0, "error": str(e)[:120]}

BACKENDS = {"local": call_local, "hosted": call_hosted}

## 4. Prompts: system vs user, few-shot, versioning

- **System prompt**: the standing rules (role, schema, "JSON only"). Changes rarely.
- **User prompt**: the specific input for this call (the transcript).
- **Few-shot**: show the model a worked example as earlier chat turns, which usually improves format reliability, especially for small models.
- **Versioning**: every prompt gets an id (`v1`, `v2`, ...). Never edit a version in place; add a new one so results stay comparable.

In [ ]:
SCHEMA_TEXT = """{
  "summary": "one sentence",
  "decisions": [{"decision": "what was decided", "owner": "person or null"}],
  "action_items": [{"owner": "person", "task": "what to do", "due": "deadline or null"}],
  "open_questions": ["unresolved question"]
}"""

PROMPTS = {
  "v1": {  # naive baseline
    "note": "Zero-shot, minimal instructions",
    "system": "You extract decisions and action items from meeting transcripts. Reply in JSON.",
    "few_shot": False,
  },
  "v2": {  # explicit schema + rules
    "note": "Zero-shot with explicit schema and rules",
    "system": ("You are a meeting analyst. Read the transcript and reply with ONLY one JSON object, no extra text, "
               "matching this schema:\n" + SCHEMA_TEXT + "\nRules: a decision is something agreed on; an action item "
               "has a named owner; ignore small talk; use null if unknown; never invent facts."),
    "few_shot": False,
  },
  "v3": {  # v2 + one worked example
    "note": "v2 rules plus one few-shot example",
    "system": ("You are a meeting analyst. Read the transcript and reply with ONLY one JSON object, no extra text, "
               "matching this schema:\n" + SCHEMA_TEXT + "\nRules: a decision is something agreed on; an action item "
               "has a named owner; ignore small talk; use null if unknown; never invent facts."),
    "few_shot": True,
  },
}

def user_prompt(transcript):
    return "Meeting transcript:\n" + transcript + "\n\nReturn the JSON now."

def build_messages(version, transcript):
    p = PROMPTS[version]
    msgs = [{"role": "system", "content": p["system"]}]
    if p["few_shot"]:
        msgs += [{"role": "user", "content": user_prompt(EXAMPLE_TRANSCRIPT)},
                 {"role": "assistant", "content": json.dumps(EXAMPLE_JSON)}]
    msgs.append({"role": "user", "content": user_prompt(transcript)})
    return msgs

# Save the prompt registry next to the notebook so versions are tracked (commit it to git!)
with open("prompts_registry.json", "w") as f:
    json.dump(PROMPTS, f, indent=2)
print("Prompt versions:", {k: v["note"] for k, v in PROMPTS.items()})

Prompt versions: {'v1': 'Zero-shot, minimal instructions', 'v2': 'Zero-shot with explicit schema and rules', 'v3': 'v2 rules plus one few-shot example'}


## 5. Parsing, validating and scoring

Small models often wrap JSON in prose or code fences. We measure two things:
- **strict JSON**: the whole reply parses as JSON (what a production pipeline wants)
- **lenient JSON**: we can rescue a JSON object from the reply
- **schema valid**: the right keys and types are present

In [8]:
def parse_output(text):
    """Returns (obj_or_None, strict_ok)."""
    text = text.strip()
    try:
        return json.loads(text), True
    except Exception:
        pass
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1]), False
        except Exception:
            pass
    return None, False

def validate(obj):
    """Returns a list of problems (empty list = valid)."""
    if not isinstance(obj, dict):
        return ["not a JSON object"]
    errs = []
    if not isinstance(obj.get("summary"), str):
        errs.append("summary must be a string")
    if not isinstance(obj.get("decisions"), list):
        errs.append("decisions must be a list")
    else:
        for d in obj["decisions"]:
            if not (isinstance(d, dict) and isinstance(d.get("decision"), str)):
                errs.append("each decision needs a 'decision' string"); break
    if not isinstance(obj.get("action_items"), list):
        errs.append("action_items must be a list")
    else:
        for a in obj["action_items"]:
            if not (isinstance(a, dict) and isinstance(a.get("owner"), str) and isinstance(a.get("task"), str)):
                errs.append("each action item needs 'owner' and 'task' strings"); break
    if not isinstance(obj.get("open_questions"), list):
        errs.append("open_questions must be a list")
    return errs

def _hit(gold_keywords, texts):
    return any(all(k in t for k in gold_keywords) for t in texts)

def score(obj, meeting):
    """Recall of gold decisions/actions via keyword match. Also counts extracted items."""
    if not isinstance(obj, dict) or validate(obj):
        return {"dec_recall": 0.0, "act_recall": 0.0, "n_items": 0}
    dec_txt = [str(d.get("decision", "")).lower() for d in obj["decisions"] if isinstance(d, dict)]
    act_txt = [(str(a.get("owner", "")) + " " + str(a.get("task", "")) + " " + str(a.get("due", ""))).lower()
               for a in obj["action_items"] if isinstance(a, dict)]
    gd, ga = meeting["gold_decisions"], meeting["gold_actions"]
    return {"dec_recall": sum(_hit(g, dec_txt) for g in gd) / len(gd),
            "act_recall": sum(_hit(g, act_txt) for g in ga) / len(ga),
            "n_items": len(dec_txt) + len(act_txt)}

# --- Unit tests that need NO model: they test our own checking code ---
good = {"summary": "s", "decisions": [{"decision": "Use Postgres", "owner": None}],
        "action_items": [{"owner": "Bob", "task": "write migration plan", "due": "Wednesday"}], "open_questions": []}
assert validate(good) == []
assert validate({"summary": "x"}) != []
assert parse_output('Sure! ```json\n{"a": 1}\n``` hope that helps')[0] == {"a": 1}
assert parse_output('{"a": 1}')[1] is True
assert parse_output("no json here")[0] is None
assert score(good, MEETINGS["product_sync"])["dec_recall"] == 0.5
print("Helper tests passed.")

Helper tests passed.


In [9]:
RESULTS = []   # every experiment run is appended here

def run_case(experiment, backend, version, meeting_name, level="short", temperature=0.0, top_p=1.0, seed=0):
    m = MEETINGS[meeting_name]
    msgs = build_messages(version, build_context(m["transcript"], level))
    r = BACKENDS[backend](msgs, temperature=temperature, top_p=top_p, seed=seed)
    obj, strict = parse_output(r["text"])
    errs = validate(obj) if obj is not None else ["unparseable"]
    s = score(obj, m)
    row = {"experiment": experiment, "backend": backend, "version": version, "meeting": meeting_name,
           "level": level, "temp": temperature, "top_p": top_p, "seed": seed,
           "strict_json": strict, "lenient_json": obj is not None, "schema_ok": not errs,
           "recall": (s["dec_recall"] + s["act_recall"]) / 2, "n_items": s["n_items"],
           "latency_s": r["latency_s"], "prompt_tokens": r["prompt_tokens"],
           "completion_tokens": r["completion_tokens"], "error": r["error"], "raw": r["text"]}
    RESULTS.append(row)
    return row

def summarize(experiment, group_keys):
    groups = defaultdict(list)
    for r in RESULTS:
        if r["experiment"] == experiment and not r["error"]:
            groups[tuple(r[k] for k in group_keys)].append(r)
    mean = lambda rows, k: sum(float(x[k]) for x in rows) / len(rows)
    head = " | ".join(f"{k:>8s}" for k in group_keys) + " |  runs | strict | schema | recall | latency_s | prompt_tok"
    print(head); print("-" * len(head))
    for key, rows in groups.items():
        print(" | ".join(f"{str(v):>8s}" for v in key) +
              f" | {len(rows):5d} | {mean(rows,'strict_json'):6.0%} | {mean(rows,'schema_ok'):6.0%} | "
              f"{mean(rows,'recall'):6.0%} | {mean(rows,'latency_s'):9.1f} | {mean(rows,'prompt_tokens'):10.0f}")
    print()

## 6. One extraction end to end

In [10]:
row = run_case("demo", "local", "v3", "product_sync")
print(row["raw"])
print("\nstrict JSON:", row["strict_json"], "| schema ok:", row["schema_ok"],
      "| recall:", f"{row['recall']:.0%}", "| latency:", f"{row['latency_s']:.1f}s")

W1018 16:06:16.682000 6112 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:01<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct in 174.5s
{"summary": "Launch date changed from November 1 to November 15.", "decisions": [{"decision": "Launch date changes to November 15", "owner": "Alice"}, {"decision": "Legal review needed for new terms", "owner": "Alice"}], "action_items": [{"owner": "Alice", "task": "Email legal team", "due": "next Wednesday"}], "open_questions": ["Do we need legal review for the new terms?", "Who will send the email?"]}

strict JSON: True | schema ok: True | recall: 42% | latency: 133.4s


## 7. Experiments

### 7a. Prompt versions (system vs user prompt design, few-shot)
Same model, temperature 0, only the prompt changes.

In [11]:
for v in ["v1", "v2", "v3"]:
    for m in MEETINGS:
        run_case("prompt_versions", "local", v, m)
summarize("prompt_versions", ["version"])

 version |  runs | strict | schema | recall | latency_s | prompt_tok
--------------------------------------------------------------------
      v1 |     2 |     0% |     0% |     0% |      90.7 |        182
      v2 |     2 |   100% |   100% |    25% |     104.3 |        293
      v3 |     2 |   100% |   100% |    33% |     136.8 |        438



### 7b. Temperature and top-p
Repeated runs with different seeds. Watch **JSON validity** drop as randomness rises.

In [12]:
settings = [(0.0, 1.0), (0.7, 0.9), (1.2, 1.0)] if QUICK else [(0.0, 1.0), (0.3, 0.9), (0.7, 0.9), (1.0, 1.0), (1.3, 1.0)]
seeds = [1, 2, 3] if QUICK else [1, 2, 3, 4, 5]
for temp, top_p in settings:
    for seed in (seeds if temp > 0 else [0]):
        run_case("sampling", "local", "v3", "product_sync", temperature=temp, top_p=top_p, seed=seed)
summarize("sampling", ["temp", "top_p"])

    temp |    top_p |  runs | strict | schema | recall | latency_s | prompt_tok
-------------------------------------------------------------------------------
     0.0 |      1.0 |     1 |   100% |   100% |    42% |     145.6 |        453
     0.7 |      0.9 |     3 |   100% |   100% |    50% |     150.8 |        453
     1.2 |      1.0 |     3 |    67% |     0% |     0% |     139.2 |        453



### 7c. Context length
We bury the real meeting inside off-topic chatter. Longer context means more tokens, more latency and a greater chance of missing items.

In [13]:
levels = ["short", "medium"] if QUICK else ["short", "medium", "long"]
for lv in levels:
    for m in MEETINGS:
        run_case("context", "local", "v3", m, level=lv)
summarize("context", ["level"])

   level |  runs | strict | schema | recall | latency_s | prompt_tok
--------------------------------------------------------------------
   short |     2 |   100% |   100% |    33% |     135.1 |        438
  medium |     2 |   100% |   100% |    12% |     212.8 |        744



### 7d. Hosted vs local
Local: a 0.5B model on your machine. Hosted: a 7B model on Hugging Face's free tier. Set `HF_TOKEN` to enable.
Note this compares **two different model sizes**, which is what a beginner actually chooses between.

In [14]:
for backend in ["local", "hosted"]:
    for m in MEETINGS:
        run_case("hosted_vs_local", backend, "v3", m)
summarize("hosted_vs_local", ["backend"])
skipped = [r for r in RESULTS if r["experiment"] == "hosted_vs_local" and r["error"]]
if skipped:
    print("Hosted skipped/failed:", skipped[0]["error"])

 backend |  runs | strict | schema | recall | latency_s | prompt_tok
--------------------------------------------------------------------
   local |     2 |   100% |   100% |    33% |     153.3 |        438

Hosted skipped/failed: no HF_TOKEN


## 8. Latency and cost

Local runs cost **$0 in API fees** (you pay in time and electricity). Hosted free tiers cost $0 but have rate limits.
To understand what you are saving, we price the same tokens at an **illustrative** paid rate. Edit the numbers; they are not real quotes.

In [15]:
PRICE_IN_PER_1M, PRICE_OUT_PER_1M = 0.50, 1.50   # hypothetical $ per 1M tokens, for illustration only

def reference_cost(rows):
    return sum(r["prompt_tokens"] * PRICE_IN_PER_1M + r["completion_tokens"] * PRICE_OUT_PER_1M for r in rows) / 1e6

for backend in ["local", "hosted"]:
    rows = [r for r in RESULTS if r["backend"] == backend and not r["error"]]
    if not rows:
        continue
    toks = sum(r["completion_tokens"] for r in rows); secs = sum(r["latency_s"] for r in rows)
    per_meeting = reference_cost(rows) / len(rows)
    print(f"{backend:7s}: {len(rows):3d} runs | {toks / max(secs, 1e-9):6.1f} output tok/s | "
          f"actual fees $0.00 | illustrative paid cost ${per_meeting:.5f}/meeting "
          f"(~${per_meeting * 10000:,.2f} per 10k meetings)")

print("\nTakeaway: prompt tokens dominate cost. Few-shot examples and long transcripts are paid for on every call.")

local  :  20 runs |    0.8 output tok/s | actual fees $0.00 | illustrative paid cost $0.00038/meeting (~$3.78 per 10k meetings)

Takeaway: prompt tokens dominate cost. Few-shot examples and long transcripts are paid for on every call.


## 9. Prompt regression tests

Before adopting a new prompt version, it must clear a bar on the same fixed test set. This guards against "improvements" that quietly break things.

In [16]:
THRESHOLDS = {"schema_ok": 0.5, "recall": 0.5}   # tune upward as your prompt/model improves

def regression_test(version, backend="local"):
    rows = [run_case("regression", backend, version, m) for m in MEETINGS]
    schema = sum(r["schema_ok"] for r in rows) / len(rows)
    recall = sum(r["recall"] for r in rows) / len(rows)
    passed = schema >= THRESHOLDS["schema_ok"] and recall >= THRESHOLDS["recall"]
    print(f"{version}: schema_ok={schema:.0%} recall={recall:.0%} -> {'PASS' if passed else 'FAIL'}")
    return passed

results = {v: regression_test(v) for v in PROMPTS}

# Candidate for the next release: change the prompt, give it a NEW version id, test it, then decide.
PROMPTS["v4"] = {**PROMPTS["v3"], "note": "v3 + stricter wording: 'Output must start with { and end with }'",
                 "system": PROMPTS["v3"]["system"] + "\nYour output must start with { and end with }."}
regression_test("v4")
with open("prompts_registry.json", "w") as f:
    json.dump(PROMPTS, f, indent=2)

v1: schema_ok=0% recall=0% -> FAIL
v2: schema_ok=100% recall=25% -> FAIL
v3: schema_ok=100% recall=33% -> FAIL
v4: schema_ok=100% recall=46% -> FAIL


## 10. Use it on your own meeting

In [26]:
MY_TRANSCRIPT = """Rohan: We need to launch the new website next week.
Sita: The homepage design is ready.
Rohan: Great, let's use the blue theme.
Anil: I'll finish the homepage by Friday.
Rohan: Please start tomorrow and update me on Monday.
Sita: I'll send the final design today.
Anil: I'll check the budget with finance tomorrow.
Rohan: Should we buy professional photos or use stock images?"""

def extract(transcript, version="v3", backend="local"):
    r = BACKENDS[backend](build_messages(version, transcript), temperature=0.0)
    obj, strict = parse_output(r["text"])
    errs = validate(obj) if obj else ["unparseable"]
    print(json.dumps(obj, indent=2) if obj else r["text"])
    print(f"\nstrict={strict} valid={not errs} {errs or ''} | {r['latency_s']:.1f}s")
    return obj

extract(MY_TRANSCRIPT)

{"summary": "Rohan needs to launch the new website next week.", "decisions": [{"decision": "Launch the new website next week", "owner": "Rohan"}], "action_items": [{"owner": "Sita", "task": "Use the blue theme", "due": "next week"}, {"owner": "Anil", "task": "Finish the homepage by Friday", "due": "Monday"}, {"owner": "Rohan", "task": "Start tomorrow and update me on Monday", "due": "Monday"}, {"owner": "Sita", "task": "Send the final design today", "due": "next week"}, {"owner": "Anil", "task": "Check the budget with finance tomorrow", "due": "next week"}], "open_questions": ["Should we buy professional photos or use stock images?"}

strict=False valid=False ['unparseable'] | 231.1s


## Cheat sheet

| Concept | Rule of thumb for extraction |
|---|---|
| Temperature | 0 to 0.3. Higher temperature lowers JSON validity. |
| top_p | Leave at 1.0 (or 0.9) when temperature is low. |
| System prompt | Put schema and rules here. Keep it stable. |
| User prompt | Only the transcript and the request. |
| Few-shot | One good example is often the biggest accuracy and validity win. |
| Context length | Cost and latency grow with tokens; accuracy can drop when the answer is buried. |
| Local vs hosted | Local: private, $0, slower, smaller models. Hosted: stronger models, rate-limited. |
| Versioning | New id for every prompt change; run regression tests before switching. |

**Limits of this notebook:** scoring is keyword-based recall (it does not penalise invented items much), the test set is tiny, and a 0.5B model is weak, so treat numbers as a teaching demo, not a benchmark. Try `Qwen/Qwen2.5-1.5B-Instruct` locally for a clear quality jump.